In [0]:
# Importer les tables Silver
from pyspark.sql.functions import (
    coalesce,
    col,
    count,
    greatest,
    lit,
    max as spark_max,
    round,
    sum as spark_sum,
    when
)

stock_items = spark.table(
    "retail_dev.silver.stock_items"
)

stock_holdings = spark.table(
    "retail_dev.silver.stock_item_holdings"
)

stock_events = spark.table(
    "retail_dev.silver.stock_events"
)

suppliers = spark.table(
    "retail_dev.silver.suppliers"
)

In [0]:
# Résumer les événements par produit
event_summary = (
    stock_events
    .groupBy("stock_item_id")
    .agg(
        spark_sum("quantity_change")
        .alias("net_event_change"),

        spark_sum(
            when(
                col("event_type") == "SALE",
                -col("quantity_change")
            ).otherwise(0)
        ).alias("units_sold_30d"),

        count("*").alias("event_count_30d"),

        spark_max("event_timestamp")
        .alias("last_event_at")
    )
)

display(event_summary.limit(20))

In [0]:
# Construire inventory_health
inventory_health = (
    stock_items.alias("i")

    .join(
        stock_holdings.alias("h"),
        col("i.stock_item_id")
        == col("h.stock_item_id"),
        "left"
    )

    .join(
        suppliers.alias("s"),
        col("i.supplier_id")
        == col("s.supplier_id"),
        "left"
    )

    .join(
        event_summary.alias("e"),
        col("i.stock_item_id")
        == col("e.stock_item_id"),
        "left"
    )

    .select(
        col("i.stock_item_id"),
        col("i.stock_item_name"),
        col("i.supplier_id"),
        col("s.supplier_name"),

        col("h.quantity_on_hand")
        .cast("long")
        .alias("initial_quantity_on_hand"),

        col("h.reorder_level")
        .cast("long")
        .alias("reorder_level"),

        col("h.target_stock_level")
        .cast("long")
        .alias("target_stock_level"),

        col("h.last_cost_price")
        .alias("unit_cost_price"),

        coalesce(
            col("e.net_event_change"),
            lit(0)
        ).cast("long").alias("net_event_change"),

        coalesce(
            col("e.units_sold_30d"),
            lit(0)
        ).cast("long").alias("units_sold_30d"),

        coalesce(
            col("e.event_count_30d"),
            lit(0)
        ).alias("event_count_30d"),

        col("e.last_event_at")
    )

    .withColumn(
        "estimated_quantity_on_hand",
        col("initial_quantity_on_hand")
        + col("net_event_change")
    )

    .withColumn(
        "average_daily_sales",
        round(
            col("units_sold_30d") / lit(30.0),
            2
        )
    )

    .withColumn(
        "days_of_cover",
        when(
            col("average_daily_sales") > 0,
            round(
                col("estimated_quantity_on_hand")
                / col("average_daily_sales"),
                2
            )
        )
    )

    .withColumn(
        "recommended_reorder_quantity",
        greatest(
            lit(0),
            col("target_stock_level")
            - col("estimated_quantity_on_hand")
        )
    )

    .withColumn(
        "inventory_value",
        round(
            greatest(
                lit(0),
                col("estimated_quantity_on_hand")
            ) * col("unit_cost_price"),
            2
        )
    )

    .withColumn(
        "stock_status",
        when(
            col("estimated_quantity_on_hand") <= 0,
            "OUT_OF_STOCK"
        )
        .when(
            col("estimated_quantity_on_hand")
            <= col("reorder_level"),
            "LOW_STOCK"
        )
        .when(
            col("estimated_quantity_on_hand")
            >= col("target_stock_level") * 1.5,
            "OVERSTOCK"
        )
        .otherwise("HEALTHY")
    )
)

In [0]:
# Enregistrer la table Gold
(
    inventory_health.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_dev.gold.inventory_health"
    )
)

print(
    "Table créée : "
    "retail_dev.gold.inventory_health"
)

In [0]:
# Afficher les résultats 
display(
    spark.table(
        "retail_dev.gold.inventory_health"
    )
    .orderBy(
        col("recommended_reorder_quantity").desc()
    )
)

In [0]:
# Résumé par état du stock 
display(
    spark.table(
        "retail_dev.gold.inventory_health"
    )
    .groupBy("stock_status")
    .agg(
        count("*").alias("product_count"),
        spark_sum("inventory_value")
        .alias("total_inventory_value"),
        spark_sum("recommended_reorder_quantity")
        .alias("total_reorder_quantity")
    )
    .orderBy("stock_status")
)